# Sổ tay dùng thuật toán cây hồi quy cộng tính Bartz
Cây hồi quy cộng tính Bayes (Bartz) là một mô hình khác xây dựng các tập hợp cây thay vì các mô hình XGB, Cat, LGBM phổ biến. Thay vì sử dụng cây tăng cường gradient, nó sử dụng cây lấy mẫu hậu nghiệm Bayes. Điều này tạo ra một mô hình đa dạng mới có thể giúp ích cho tập hợp dự đoán Tỷ lệ khách hàng rời bỏ của nónó

Chúng tôi kết hợp mô hình này với các mô hình XGBoost, Hồi quy Logistic với TA, PyTorch MLP [tại đây][1], GNN [tại đây][2] trước đây và nhận thấy rằng mô hình này đa dạng hơn rất nhiềunhiều

[1]: https://www.kaggle.com/code/cdeotte/chatgpt-vibe-coding-3xgpu-models-cv-0-9178
[2]: https://www.kaggle.com/code/cdeotte/gnn-starter-cv-0-9155-with-hill-climbing-demo

# Pip install Bartz

In [1]:
!pip install bartz==0.4.1

# Train Bartz

In [ ]:
import os
import gc
import warnings
from itertools import combinations

import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import roc_auc_score

warnings.simplefilter("ignore")

# ============================================================
# tinh chỉnhchỉnh
# ============================================================
SEED = 42
N_SPLITS = 5
VER = 8400

TRAIN_PATH = "/kaggle/input/competitions/playground-series-s6e3/train.csv"
TEST_PATH = "/kaggle/input/competitions/playground-series-s6e3/test.csv"
ORIG_PATH = "/kaggle/input/datasets/cdeotte/s6e3-original-dataset/WA_Fn-UseC_-Telco-Customer-Churn.csv"

TARGET = "Churn"
ID_COL = "id"
ORIG_ID_COL = "customerID"

CATS = [
    "gender",
    "SeniorCitizen",
    "Partner",
    "Dependents",
    "PhoneService",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaperlessBilling",
    "PaymentMethod",
]

NUMS = ["tenure", "MonthlyCharges", "TotalCharges"]

BARTZ_PARAMS = {
    "maxdepth": 6,
    "ntree": 400,
    "k": 5,
    "sigdf": 3,
    "sigquant": 0.9,
}

NDPOST = 1000
NSKIP = 200
KEEPEVERY = 2

# ============================================================
# INSTALL / IMPORT BARTZ
# ============================================================
try:
    os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
    from bartz.BART import gbart
except Exception as e:
    raise ImportError(
        "bartz is required.\n"
        "Install with:\n"
        "  pip install bartz==0.4.1\n"
        f"Original import error: {e}"
    )

# ============================================================
# HELPERS
# ============================================================
def set_seed(seed=42):
    np.random.seed(seed)

def safe_total_charges(series: pd.Series) -> pd.Series:
    # Nhà cung cấp dịch vụ viễn thông gốc thường có chuỗi ký tự trống " "
    return pd.to_numeric(
        series.astype(str).str.strip().replace("", np.nan),
        errors="coerce"
    ).astype("float32")

def map_target(series: pd.Series) -> pd.Series:
    if series.dtype == "O":
        return series.map({"Yes": 1, "No": 0}).astype("int8")
    return series.astype("int8")

# target encoder 
class TargetEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, cols_to_encode, aggs=["mean"], cv=5, smooth="auto", drop_original=False):
        self.cols_to_encode = cols_to_encode
        self.aggs = aggs
        self.cv = cv
        self.smooth = smooth
        self.drop_original = drop_original
        self.mappings_ = {}
        self.global_stats_ = {}

    def fit(self, X, y):
        temp_df = X.copy()
        temp_df["target"] = y

        for agg_func in self.aggs:
            self.global_stats_[agg_func] = y.agg(agg_func)

        for col in self.cols_to_encode:
            self.mappings_[col] = {}
            for agg_func in self.aggs:
                mapping = temp_df.groupby(col)["target"].agg(agg_func)
                self.mappings_[col][agg_func] = mapping
        return self

    def transform(self, X):
        X_transformed = X.copy()
        for col in self.cols_to_encode:
            for agg_func in self.aggs:
                new_col_name = f"TE_{col}_{agg_func}"
                map_series = self.mappings_[col][agg_func]
                X_transformed[new_col_name] = X[col].map(map_series)
                X_transformed[new_col_name] = X_transformed[new_col_name].fillna(
                    self.global_stats_[agg_func]
                )

        if self.drop_original:
            X_transformed.drop(columns=self.cols_to_encode, inplace=True)

        return X_transformed

    def fit_transform(self, X, y):
        self.fit(X, y)

        encoded_features = pd.DataFrame(index=X.index)
        kf = KFold(n_splits=self.cv, shuffle=True, random_state=42)

        for train_idx, val_idx in kf.split(X, y):
            X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
            X_val = X.iloc[val_idx]

            temp_df_train = X_train.copy()
            temp_df_train["target"] = y_train

            for col in self.cols_to_encode:
                for agg_func in self.aggs:
                    new_col_name = f"TE_{col}_{agg_func}"

                    fold_global_stat = y_train.agg(agg_func)
                    mapping = temp_df_train.groupby(col)["target"].agg(agg_func)

                    if agg_func == "mean":
                        counts = temp_df_train.groupby(col)["target"].count()

                        m = self.smooth
                        if self.smooth == "auto":
                            variance_between = mapping.var()
                            avg_variance_within = temp_df_train.groupby(col)["target"].var().mean()
                            if pd.notna(variance_between) and variance_between > 0:
                                m = avg_variance_within / variance_between
                                if pd.isna(m):
                                    m = 0
                            else:
                                m = 0

                        smoothed_mapping = (counts * mapping + m * fold_global_stat) / (counts + m)
                        encoded_values = X_val[col].map(smoothed_mapping)
                    else:
                        encoded_values = X_val[col].map(mapping)

                    encoded_features.loc[X_val.index, new_col_name] = encoded_values.fillna(
                        fold_global_stat
                    )

        X_transformed = X.copy()
        for col in encoded_features.columns:
            X_transformed[col] = encoded_features[col]

        if self.drop_original:
            X_transformed.drop(columns=self.cols_to_encode, inplace=True)

        return X_transformed

def add_digit_cols(df, col, multiplier, prefix_list=None):
    """
    - Chuyển đổi dạng số sang dạng biểu diễn giống số nguyên
    - Tách thành các chữ số chuỗi có độ dài tối đa          
    """
    temp_col = f"{col}_TEMP_INT"

    if multiplier == "direct":
        vals = pd.to_numeric(df[col], errors="coerce").fillna(-1).astype(int)
    else:
        vals = (pd.to_numeric(df[col], errors="coerce") * multiplier).round(0).fillna(-1).astype(int)

    df[temp_col] = vals

    temp_str = df[temp_col].astype(str)

    if col == "tenure":
        max_len = 3
    elif col == "MonthlyCharges":
        max_len = 5   # e.g. 118.75 * 100 = 11875 (quá nhiềunhiều)
    elif col == "TotalCharges":
        max_len = 6   # enough for churn
    else:
        max_len = temp_str.str.len().max()

    temp_str_padded = temp_str.str.zfill(max_len)

    created = []
    for i in range(max_len):
        new_col = f"{col}_DIGIT_{i+1}"
        df[new_col] = temp_str_padded.str[i].astype(int)
        created.append(new_col)
        if prefix_list is not None:
            prefix_list.append(new_col)

    df.drop(columns=[temp_col], inplace=True)
    return created

def factorize_together(train_s, val_s, test_s):
    combined = pd.concat([train_s, val_s, test_s], axis=0, ignore_index=True)
    codes, _ = pd.factorize(combined.astype(str))
    n1 = len(train_s)
    n2 = len(val_s)
    train_codes = pd.Series(codes[:n1], index=train_s.index, dtype="int32")
    val_codes = pd.Series(codes[n1:n1+n2], index=val_s.index, dtype="int32")
    test_codes = pd.Series(codes[n1+n2:], index=test_s.index, dtype="int32")
    return train_codes, val_codes, test_codes


# ============================================================
# LOAD DATA
# ============================================================
set_seed(SEED)

print("Loading data...")
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
orig = pd.read_csv(ORIG_PATH)

print("train:", train.shape)
print("test :", test.shape)
print("orig :", orig.shape)

train_ids = train[ID_COL].copy() if ID_COL in train.columns else pd.Series(np.arange(len(train)), name="id")
test_ids = test[ID_COL].copy() if ID_COL in test.columns else pd.Series(np.arange(len(test)), name="id")

if ID_COL in train.columns:
    train = train.drop(columns=[ID_COL])
if ID_COL in test.columns:
    test = test.drop(columns=[ID_COL])
if ORIG_ID_COL in orig.columns:
    orig = orig.drop(columns=[ORIG_ID_COL])

train[TARGET] = map_target(train[TARGET])
if TARGET in orig.columns:
    orig[TARGET] = map_target(orig[TARGET])

# dọn dẹp số an toàn
for df in [train, test, orig]:
    df["tenure"] = pd.to_numeric(df["tenure"], errors="coerce").astype("float32")
    df["MonthlyCharges"] = pd.to_numeric(df["MonthlyCharges"], errors="coerce").astype("float32")
    df["TotalCharges"] = safe_total_charges(df["TotalCharges"])

    # Cách khắc phục lỗi bỏ ngang thực tế tương tự như các quy trình khác của bạn
    fill_val = df["tenure"] * df["MonthlyCharges"]
    df["TotalCharges"] = df["TotalCharges"].fillna(fill_val).astype("float32")

# dọn dẹp theo danh mục
for df in [train, test, orig]:
    for c in CATS:
        df[c] = df[c].astype(str).fillna("Missing")

BASE = [c for c in train.columns if c not in [TARGET]]
print("BASE cols:", len(BASE))

# ============================================================
# DIGIT FEATURES
# ============================================================
DIGIT = []

for df in [train, test, orig]:
    add_digit_cols(df, "tenure", "direct", DIGIT if df is train else None)
    add_digit_cols(df, "MonthlyCharges", 100, DIGIT if df is train else None)
    add_digit_cols(df, "TotalCharges", 1, DIGIT if df is train else None)

DIGIT = list(dict.fromkeys(DIGIT))
print(f"{len(DIGIT)} DIGIT features created.")

# ============================================================
# ROUND FEATURES
# ============================================================
ROUND = []

rounding_levels = {
    "1s": 0,
    "10s": -1,
    "100s": -2,
    "1000s": -3,
}

for col in ["MonthlyCharges", "TotalCharges"]:
    for suffix, level in rounding_levels.items():
        new_col = f"{col}_ROUND_{suffix}"
        ROUND.append(new_col)
        for df in [train, test, orig]:
            df[new_col] = df[col].round(level).fillna(-1).astype(int)

# add tenure rounds too, since churn has only 3 numerics
for suffix, level in {
    "1s": 0,
    "10s": -1,
}.items():
    new_col = f"tenure_ROUND_{suffix}"
    ROUND.append(new_col)
    for df in [train, test, orig]:
        df[new_col] = df["tenure"].round(level).fillna(-1).astype(int)

print(f"{len(ROUND)} ROUND features created.")

# ============================================================
# INTERACTION FEATURES
# ============================================================
INTER = []

print("Building pairwise interaction features...")
for col1, col2 in combinations(BASE, 2):
    new_col = f"{col1}_{col2}"
    INTER.append(new_col)
    for df in [train, test, orig]:
        df[new_col] = df[col1].astype(str) + "_" + df[col2].astype(str)

print(f"{len(INTER)} interaction features created.")

# ============================================================
# ORIG FEATURES
# ============================================================
ORIG = []
orig_target_mean = float(orig[TARGET].mean())

print("Building orig mean/count features...")
for col in BASE:
    mean_map = orig.groupby(col)[TARGET].mean()
    mean_name = f"orig_mean_{col}"
    mean_map.name = mean_name

    train = train.merge(mean_map, on=col, how="left")
    test = test.merge(mean_map, on=col, how="left")
    ORIG.append(mean_name)

    count_name = f"orig_count_{col}"
    count_map = orig.groupby(col).size().reset_index(name=count_name)

    train = train.merge(count_map, on=col, how="left")
    test = test.merge(count_map, on=col, how="left")
    ORIG.append(count_name)

train[ORIG] = train[ORIG].fillna(orig_target_mean)
test[ORIG] = test[ORIG].fillna(orig_target_mean)

print(f"{len(ORIG)} ORIG features created.")

# ============================================================
# FEATURES
# ============================================================
FEATURES = BASE + ORIG + INTER + ROUND + DIGIT
FEATURES = list(dict.fromkeys(FEATURES))

print(f"{len(FEATURES)} total features.")

X = train[FEATURES].copy()
y = train[TARGET].copy()

BASE_TE_COL = ["tenure", "MonthlyCharges", "TotalCharges"] + ROUND + DIGIT

# ============================================================
# CV
# ============================================================
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

oof_preds = np.zeros(len(X), dtype=np.float32)
test_preds = np.zeros(len(test), dtype=np.float32)

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    print(f"\n--- Fold {fold}/{N_SPLITS} ---")

    X_train = X.iloc[train_idx].copy()
    X_val = X.iloc[val_idx].copy()
    y_train = y.iloc[train_idx].copy()
    y_val = y.iloc[val_idx].copy()
    X_test = test[FEATURES].copy()

    # Tương tác TE, sau đó loại bỏ các chuỗi tương tác gốc
    te_inter = TargetEncoder(
        cols_to_encode=INTER,
        cv=5,
        smooth="auto",
        aggs=["mean"],
        drop_original=True
    )
    X_train = te_inter.fit_transform(X_train, y_train)
    X_val = te_inter.transform(X_val)
    X_test = te_inter.transform(X_test)

    # TE đã chọn các cột cơ bản, giữ nguyên các cột gốc
    te_base = TargetEncoder(
        cols_to_encode=BASE_TE_COL,
        cv=5,
        smooth="auto",
        aggs=["mean"],
        drop_original=False
    )
    X_train = te_base.fit_transform(X_train, y_train)
    X_val = te_base.transform(X_val)
    X_test = te_base.transform(X_test)

    for c in CATS:
        X_train[c], X_val[c], X_test[c] = factorize_together(
            X_train[c], X_val[c], X_test[c]
        )

    # Đối với bất kỳ cột nào còn lại không phải là số -> cũng phải phân tích thành thừa số để đảm bảo an toàn.
    for c in X_train.columns:
        if not pd.api.types.is_numeric_dtype(X_train[c]):
            X_train[c], X_val[c], X_test[c] = factorize_together(
                X_train[c], X_val[c], X_test[c]
            )

    # bartz kỳ vọng có các tính năng x mẫu
    model = gbart(
        X_train.to_numpy(dtype=np.float32).T,
        y_train.to_numpy(dtype=np.float32),
        **BARTZ_PARAMS,
        ndpost=NDPOST,
        nskip=NSKIP,
        keepevery=KEEPEVERY,
    )

    val_preds = model.predict(X_val.to_numpy(dtype=np.float32).T).mean(axis=0)
    oof_preds[val_idx] = val_preds.astype(np.float32)

    fold_auc = roc_auc_score(y_val, val_preds)
    print(f"Fold {fold} AUC: {fold_auc:.6f}")

    fold_test = model.predict(X_test.to_numpy(dtype=np.float32).T).mean(axis=0)
    test_preds += fold_test.astype(np.float32) / N_SPLITS

    del X_train, X_val, X_test, y_train, y_val, model, val_preds, fold_test
    gc.collect()

overall_auc = roc_auc_score(y, oof_preds)
print("\n====================")
print(f"Overall OOF AUC: {overall_auc:.6f}")
print("====================")

# optional clipping
oof_preds = np.clip(oof_preds, 0, 1)
test_preds = np.clip(test_preds, 0, 1)

# ============================================================
# SAVE
# ============================================================
oof_npy = f"oof_bartz_churn_{VER}.npy"
pred_npy = f"pred_bartz_churn_{VER}.npy"

np.save(oof_npy, oof_preds)
np.save(pred_npy, test_preds)

print("\nSaved:")
print(" -", oof_npy)
print(" -", pred_npy)

Loading data...
train: (594194, 21)
test : (254655, 20)
orig : (7043, 21)
BASE cols: 19
14 DIGIT features created.
10 ROUND features created.
Building pairwise interaction features...
171 interaction features created.
Building orig mean/count features...
38 ORIG features created.
252 total features.

--- Fold 1/5 ---
Iteration  100/2200 P_grow=0.50 P_prune=0.49 A_grow=0.21 A_prune=0.21 (burnin)
Iteration  200/2200 P_grow=0.54 P_prune=0.46 A_grow=0.19 A_prune=0.22 (burnin)
Iteration  300/2200 P_grow=0.53 P_prune=0.47 A_grow=0.19 A_prune=0.21
Iteration  400/2200 P_grow=0.49 P_prune=0.51 A_grow=0.19 A_prune=0.24
Iteration  500/2200 P_grow=0.49 P_prune=0.51 A_grow=0.15 A_prune=0.26
Iteration  600/2200 P_grow=0.53 P_prune=0.47 A_grow=0.19 A_prune=0.20
Iteration  700/2200 P_grow=0.52 P_prune=0.47 A_grow=0.21 A_prune=0.23
Iteration  800/2200 P_grow=0.53 P_prune=0.47 A_grow=0.18 A_prune=0.20
Iteration  900/2200 P_grow=0.54 P_prune=0.46 A_grow=0.20 A_prune=0.23
Iteration 1000/2200 P_grow=0.52 P

# Thuật toán leo đồi GPU
Chúng tôi chạy thuật toán leo đồi với mô hình Bartz mới cùng với XGBoost cũ, Hồi quy Logistic với TA, PyTorch MLP [tại đây][1] và GNN [tại đây][2]. Chúng tôi nhận thấy rằng thuật toán leo đồi chọn trọng số từ tất cả các mô hình, chứng tỏ Bartz rất đa dạng và mang lại lợi ích cho mô hình tổng hợp của chúng tôi, tuyệt vời!

[1]: https://www.kaggle.com/code/cdeotte/chatgpt-vibe-coding-3xgpu-models-cv-0-9178
[2]: https://www.kaggle.com/code/cdeotte/gnn-starter-cv-0-9155-with-hill-climbing-demo

In [ ]:
# ===========================================================
# TỔ HỢP LEO ĐỒI GPU (AUC, SỬ DỤNG XẾP HẠNG)
# - Chuyển đổi mọi dự đoán OOF và TEST thành xếp hạng chuẩn hóa toàn cầu
# - Kết hợp trực tiếp trong không gian xếp hạng
# - Bắt đầu từ mô hình đơn tốt nhất (hoặc FIRST nếu được cung cấp)
# - Liên tục thử thêm mọi mô hình với tìm kiếm lưới trọng số
# - Sử dụng AUC xấp xỉ GPU trong quá trình tìm kiếm, AUC chính xác của CPU để kiểm tra cuối cùng
# ============================================================

# CLEAN MEMORY #######################
import gc

# Xóa các đối tượng CPU/GPU lớn khỏi bartz nếu chúng vẫn còn tồn tại
for v in [
    "model", "X_train", "X_val", "X_test", "y_train", "y_val",
    "val_preds", "fold_test"
]:
    if v in globals():
        del globals()[v]

gc.collect()

# Xóa các khối VRAM được lưu trong bộ nhớ cache của CuPy
import cupy as cp
cp.get_default_memory_pool().free_all_blocks()
cp.get_default_pinned_memory_pool().free_all_blocks()
#####################################

import os
import gc
import numpy as np
import pandas as pd
import cupy as cp

from sklearn.metrics import roc_auc_score
from scipy.stats import rankdata

# -----------------------------
# tinh chỉnh
# -----------------------------
TRAIN_CSV  = "/kaggle/input/competitions/playground-series-s6e3/train.csv"
TEST_CSV   = "/kaggle/input/competitions/playground-series-s6e3/test.csv"
TARGET_COL = "Churn"
ID_COL     = "id"
VER        = 2

# Tùy chọn: buộc khởi động mô hình theo tên, nếu không thì sử dụng mô hình tốt nhất
FIRST = None

USE_NEGATIVE_WGT = True
MAX_MODELS = 1000
TOL = 1e-5

start = -0.50 if USE_NEGATIVE_WGT else 0.01
stop  =  0.50
step  =  0.01

# NOTE: 
SUBMISSION_PATH = f"submission_hillclimb_v{VER}.csv"
WEIGHTS_PATH    = f"ensemble_weights_v{VER}.csv"

# -----------------------------
# MODELS
# -----------------------------
FILES = [
    ("bartz_v8400",     "oof_bartz_churn_8400.npy"),
    ("gnn_v1909",     "/kaggle/input/notebooks/cdeotte/gnn-starter-cv-0-9155-with-hill-climbing-demo/oof_gnn_v1909.npy"),
    ("nn_v312",     "/kaggle/input/notebooks/cdeotte/chatgpt-vibe-coding-3xgpu-models-cv-0-9178/oof_nn_v312.npy"),
    ("xgb_v102",     "/kaggle/input/notebooks/cdeotte/chatgpt-vibe-coding-3xgpu-models-cv-0-9178/oof_pred_proba_v102.npy"),
    ("logreg_v200",     "/kaggle/input/notebooks/cdeotte/chatgpt-vibe-coding-3xgpu-models-cv-0-9178/oof_tepair_logit3_v200.npy"),
]

# -----------------------------
# HELPERS
# -----------------------------
def to_binary_y(df, col):
    y = df[col]
    if y.dtype == "object" or str(y.dtype).startswith("string"):
        y = y.map({"No": 0, "Yes": 1})
    y = pd.to_numeric(y, errors="coerce")
    if y.isna().any():
        raise ValueError(f"Could not parse target column '{col}' to binary.")
    y = y.astype(np.int32).to_numpy()
    return y

def auc_cpu(y_true, y_pred):
    return float(roc_auc_score(y_true, y_pred))

def rank01(x):
    x = np.asarray(x, dtype=np.float64).reshape(-1)
    r = rankdata(x, method="average")
    if len(r) == 1:
        return np.array([0.5], dtype=np.float64)
    return (r - 1.0) / (len(r) - 1.0)

def derive_test_path(oof_path):
    cand1 = oof_path.replace("oof", "pred", 1)
    cand2 = oof_path.replace("oof", "test", 1)
    if os.path.exists(cand1):
        return cand1
    if os.path.exists(cand2):
        return cand2
    raise FileNotFoundError(f"Could not find pred/test file for: {oof_path}")

def load_preds_1d(path, expected_len, csv_col=TARGET_COL):
    if path.endswith(".csv"):
        df = pd.read_csv(path)
        if csv_col not in df.columns:
            raise ValueError(f"{path} missing column '{csv_col}'")
        arr = df[csv_col].to_numpy()
    else:
        arr = np.load(path)

    arr = np.asarray(arr, dtype=np.float64)

    if arr.ndim == 2:
        if arr.shape[1] == 1:
            arr = arr[:, 0]
        elif arr.shape[0] == expected_len:
            arr = arr.mean(axis=1)
        else:
            raise ValueError(f"Unexpected shape for {path}: {arr.shape}")

    arr = arr.reshape(-1)

    if len(arr) != expected_len:
        raise ValueError(f"Length mismatch for {path}: got {len(arr)}, expected {expected_len}")

    return arr

def multiple_roc_auc_scores(y_gpu, preds_gpu):
    """
    AUC xấp xỉ nhanh trên GPU cho nhiều cột ứng viên cùng một lúc.
    Các giá trị trùng lặp chỉ là xấp xỉ, điều này chấp nhận được đối với thuật toán tìm kiếm leo đồi.
    """
    n_pos = cp.sum(y_gpu)
    n_neg = y_gpu.size - n_pos
    ranks = cp.argsort(cp.argsort(preds_gpu, axis=0), axis=0) + 1
    aucs = (cp.sum(ranks[y_gpu == 1, :], axis=0) - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)
    return aucs

# ============================================================
# 1) LOAD DATA
# ============================================================
train = pd.read_csv(TRAIN_CSV)
test  = pd.read_csv(TEST_CSV)

y = to_binary_y(train, TARGET_COL)
N = len(train)
M = len(test)

print(f"Train rows: {N:,} | Pos rate: {y.mean():.6f}")
print(f"Test rows : {M:,}")

# ============================================================
# 2) LOAD OOF / TEST PREDICTIONS
# ============================================================
names = []
x_train = []
x_test = []

print("\nLoading predictions...")
for name, oof_path in FILES:
    test_path = derive_test_path(oof_path)

    oof = load_preds_1d(oof_path, N)
    pred = load_preds_1d(test_path, M)

    oof = rank01(oof)
    pred = rank01(pred)

    names.append(name)
    x_train.append(oof)
    x_test.append(pred)

    print(f"{name:18s} | {oof_path} | {test_path}")

x_train = np.column_stack(x_train)
x_test  = np.column_stack(x_test)

print("\nOOF matrix :", x_train.shape)
print("TEST matrix:", x_test.shape)

# ============================================================
# 3) SINGLE MODEL AUC
# ============================================================
print("\nSingle-model AUC:")
single_aucs = []
for k, name in enumerate(names):
    s = auc_cpu(y, x_train[:, k])
    single_aucs.append(s)
    print(f"{name:18s} AUC = {s:.6f}")

best_idx = int(np.argmax(single_aucs))
best_auc = float(single_aucs[best_idx])
print(f"\nBest single: {names[best_idx]} (AUC={best_auc:.6f})")

# ============================================================
# 4) START MODEL
# ============================================================
if FIRST is None:
    start_idx = best_idx
else:
    if FIRST not in names:
        raise ValueError(f"FIRST='{FIRST}' not found in model list.")
    start_idx = names.index(FIRST)

start_auc = auc_cpu(y, x_train[:, start_idx])
print(f"Starting from: {names[start_idx]} (AUC={start_auc:.6f})")

# ============================================================
# 5) PREP GPU
# ============================================================
x_gpu = cp.asarray(x_train, dtype=cp.float32)
y_gpu = cp.asarray(y, dtype=cp.int8)

ww = cp.arange(start, stop + 1e-9, step, dtype=cp.float32)

best_ensemble = x_gpu[:, start_idx].copy()
best_score = float(start_auc)

models = [start_idx]
weights_step = []

# ============================================================
# 6) HILL CLIMB
# ============================================================
print("\nStarting hill climb...\n")

for it in range(1_000_000):
    if len(set(models)) >= MAX_MODELS:
        print(f"Reached MAX_MODELS={MAX_MODELS}")
        break

    base = best_ensemble[:, None] * (1.0 - ww)[None, :]

    best_it_score = -1.0
    best_it_idx = -1
    best_it_w = None
    best_it_ens = None

    for k in range(len(names)):
        cand = base + x_gpu[:, k][:, None] * ww[None, :]
        aucs = multiple_roc_auc_scores(y_gpu, cand)

        j = int(cp.argmax(aucs).item())
        s = float(aucs[j].item())

        if s > best_it_score:
            best_it_score = s
            best_it_idx = k
            best_it_w = float(ww[j].item())
            best_it_ens = cand[:, j]

        del cand, aucs

    improve = best_it_score - best_score
    if improve < TOL:
        print(f"Stopped: improvement {improve:.8f} < TOL={TOL}")
        break

    print(f"{it:3d}  AUC {best_it_score:.6f} | add {names[best_it_idx]} | w = {best_it_w:.3f}")

    models.append(best_it_idx)
    weights_step.append(best_it_w)
    best_ensemble = best_it_ens
    best_score = best_it_score

    del base
    gc.collect()
    cp._default_memory_pool.free_all_blocks()

# ============================================================
# 7) FINAL EXACT OOF AUC
# ============================================================
final_oof = best_ensemble.get()
final_auc = auc_cpu(y, final_oof)
print(f"\nFinal OOF AUC (exact CPU): {final_auc:.6f}")

# ============================================================
# 8) TRỌNG SỐ BƯỚC -> TRỌNG SỐSỐ HIỆU QUẢ
# ============================================================
wgt = np.array([1.0], dtype=np.float64)
for w in weights_step:
    wgt = wgt * (1.0 - w)
    wgt = np.append(wgt, w)

dfw = (
    pd.DataFrame({
        "model": [names[m] for m in models],
        "weight": wgt
    })
    .groupby("model", as_index=False)["weight"].sum()
    .sort_values("weight", ascending=False)
    .reset_index(drop=True)
)

print("\nFinal weights:")
print(dfw.to_string(index=False))
print("\nWeight sum:", dfw["weight"].sum())

dfw.to_csv(WEIGHTS_PATH, index=False)
print(f"Saved weights: {WEIGHTS_PATH}")

# ============================================================
# 9) KIỂM TRA TÍNH CHẤT OOF
# ============================================================
blend_oof = np.zeros(N, dtype=np.float64)
for _, row in dfw.iterrows():
    k = names.index(row["model"])
    blend_oof += x_train[:, k] * float(row["weight"])

print(f"Sanity OOF AUC (exact CPU): {auc_cpu(y, blend_oof):.6f}")

# ============================================================
# 10) THỬ NGHIỆM PHA TRỘN + NỘP BÀI
# ============================================================
blend_test = np.zeros(M, dtype=np.float64)
for _, row in dfw.iterrows():
    k = names.index(row["model"])
    blend_test += x_test[:, k] * float(row["weight"])

if ID_COL in test.columns:
    sub = pd.DataFrame({ID_COL: test[ID_COL].values, TARGET_COL: blend_test})
else:
    sub = pd.DataFrame({TARGET_COL: blend_test})

sub.to_csv(SUBMISSION_PATH, index=False)
print(f"\nSaved submission: {SUBMISSION_PATH}")
print(sub.head())

print("\nDone.")

Train rows: 594,194 | Pos rate: 0.225208
Test rows : 254,655

Loading predictions...
bartz_v8400        | oof_bartz_churn_8400.npy | pred_bartz_churn_8400.npy
gnn_v1909          | /kaggle/input/notebooks/cdeotte/gnn-starter-cv-0-9155-with-hill-climbing-demo/oof_gnn_v1909.npy | /kaggle/input/notebooks/cdeotte/gnn-starter-cv-0-9155-with-hill-climbing-demo/pred_gnn_v1909.npy
nn_v312            | /kaggle/input/notebooks/cdeotte/chatgpt-vibe-coding-3xgpu-models-cv-0-9178/oof_nn_v312.npy | /kaggle/input/notebooks/cdeotte/chatgpt-vibe-coding-3xgpu-models-cv-0-9178/pred_nn_v312.npy
xgb_v102           | /kaggle/input/notebooks/cdeotte/chatgpt-vibe-coding-3xgpu-models-cv-0-9178/oof_pred_proba_v102.npy | /kaggle/input/notebooks/cdeotte/chatgpt-vibe-coding-3xgpu-models-cv-0-9178/test_pred_proba_v102.npy
logreg_v200        | /kaggle/input/notebooks/cdeotte/chatgpt-vibe-coding-3xgpu-models-cv-0-9178/oof_tepair_logit3_v200.npy | /kaggle/input/notebooks/cdeotte/chatgpt-vibe-coding-3xgpu-models-cv-0-91